In [ ]:
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import VecMonitor
from stable_baselines3.common.vec_env.base_vec_env import VecEnv as SB3VecEnv

from gym_env import BombermanVecEnv
from bots import SelfPlayOpponent, FrozenPolicy
from sim import Settings

In [ ]:
class SmallGridCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        n_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_channels, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            sample = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations))

In [ ]:
class SB3Compatible(BombermanVecEnv, SB3VecEnv):
    """BombermanVecEnv already implements the VecEnv method surface;
    this just satisfies SB3's isinstance check."""

    def __init__(self, num_envs, settings=None, opponent=None, seed=0):
        BombermanVecEnv.__init__(self, num_envs, settings, opponent, seed)

In [ ]:
class SelfPlayCallback(BaseCallback):
    """Periodically freezes the current policy into the opponent's
    snapshot pool and logs self-play metrics to TensorBoard."""

    def __init__(self, opponent: SelfPlayOpponent, snapshot_freq: int, verbose: int = 0):
        super().__init__(verbose)
        self.opponent = opponent
        self.snapshot_freq = snapshot_freq
        self._recent_wins: list[float] = []

    def _on_step(self) -> bool:
        for info, done in zip(self.locals.get("infos", []), self.locals.get("dones", [])):
            if done and "won" in info:
                self._recent_wins.append(float(info["won"]))
        self._recent_wins = self._recent_wins[-200:]

        if self._recent_wins:
            self.logger.record("selfplay/win_rate_vs_pool", float(np.mean(self._recent_wins)))
        self.logger.record("selfplay/pool_size", len(self.opponent.policies))

        if self.n_calls % self.snapshot_freq == 0:
            self.opponent.add_snapshot(self.model.policy)
            if self.verbose:
                print(f"[selfplay] snapshot #{len(self.opponent.policies)} "
                      f"added at step {self.num_timesteps}")
        return True

In [ ]:
n_envs = 2048
opponent = SelfPlayOpponent(pool_size=5)
venv = SB3Compatible(n_envs, Settings(scenario="classic"), opponent=opponent)
venv = VecMonitor(venv)

model = PPO(
    "CnnPolicy",
    venv,
    device="cuda" if torch.cuda.is_available() else "cpu",
    n_steps=512,
    batch_size=16*1024,
    learning_rate=3e-4,
    n_epochs=10,
    verbose=1,
    tensorboard_log="./runs",
    policy_kwargs=dict(
        features_extractor_class=SmallGridCNN,
        features_extractor_kwargs=dict(features_dim=256),
        normalize_images=False,
    ),
)

callback = SelfPlayCallback(opponent, snapshot_freq=50_000, verbose=1)
model.learn(total_timesteps=2_000_000, callback=callback, tb_log_name="ppo_selfplay")
model.save("bomberman_ppo")